[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2_orientation.ipynb)
# Rationale 2 Orientation: Detecting Plant Health

##### Welcome. This notebook orients you to PlantVillage, the image dataset that anchors all four Rationale 2 questions.

#### By the end you will have:

- **A Colab environment set up with the `irilab2026` library and a sample of the PlantVillage dataset cached locally.**
- **A sense of what PlantVillage is, how it was built, and why one specific fact about it (Noyan 2022) shapes most of the Rationale 2 work you'll do.**
- **An overview of lab-condition disease photographs — six to eight leaves spanning the dataset.**
- **A sense of the dataset's class structure: 38 host-disease classes, ~54,000 images total, and the class imbalance you'll need to consider.**
- **A clear handoff to Rationale 2 Question 1, where PlantDoc enters and the comparison with PlantVillage begins.**

You don't need to install anything on your computer. Everything runs in Colab. To begin, click the cells in order from top to bottom and run each one. We'll explain what each cell does as we go.

#### What It Is and Who Built It
**PlantVillage** is a research dataset containing approximately 54,000 leaf images across 38 classes, representing 14 host plant species and their diseases. It was introduced in the Mohanty, Hughes & Salathé 2016 paper that proposed using deep learning for smartphone-based plant disease diagnosis in low-resource agricultural settings. The project's main goal was to create "a tool a farmer could use in the field." 

In the lab, plant leaves were removed, placed against solid backgrounds (usually gray or black), and photographed under controlled lighting, with each image capturing a single leaf. This approach was intentional: clear, uniform images provide better training data, illustrating concretely what "lab-photographed" means. The design choices of **PlantVillage** have implications that the original paper does not address. Noyan 2022 demonstrated that a classifier trained on just eight background pixels—without any leaf—predicts PlantVillage class labels with about 49% accuracy. This is significant, considering chance performance is below 3% across 38 roughly balanced classes. This isn’t a flaw but a characteristic of the dataset—its lab-condition design and the background's predictability reflect the same underlying aspect from different perspectives. 

The questions in Rationale 2 are centered on this fact. R2-Q1 explores whether a model trained on lab-condition PlantVillage images can transfer to field photographs of the same diseases (PlantDoc). R2-Q2 investigates what the model attends to when it makes errors. R2-Q3 considers whether targeted augmentation can reduce the gap. R2-Q4 examines whether the model learns the disease or the host plant. Collectively, these questions form a coordinated effort to understand what a classifier trained on this specific dataset has actually learned. The section concludes by preparing you to analyze the dataset directly in the rest of this notebook.

## Setup
Before we look at any data, we need to install the lab's shared library, set up the working environment, and confirm that everything is in place. The `irilab2026` library bundles four small helpers that handle the chores you'd otherwise do by hand at the start of every notebook, plus a wrapper function that runs them in one shot. We'll meet the four helpers individually first, then use the wrapper.

In [ ]:
!pip install git+https://github.com/geraldmc/irilab2026.git -q

In [ ]:
import irilab2026 as iri

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

### The four environment helpers

The library exposes four functions that each handle one piece of getting a Colab notebook ready to do real work:

- `is_colab()` returns `True` if you're running in Google Colab, `False` otherwise. Useful for code that should behave differently depending on where it runs.
- `has_gpu()` returns `True` if a GPU is attached to the runtime. Image-model training is impractical without one; orientation work like this notebook doesn't need one.
- `cache_dir()` returns the directory where downloaded data will be stored. On Colab with Google Drive mounted, this is a folder on your Drive — so cached data survives a runtime reset. Locally, it's a folder in your home directory.
- `mount_google_drive()` mounts your Drive at `/content/drive` when you're on Colab, so that downloads persist. Has no effect locally. We won't call this directly here; the wrapper function in a moment will handle it for us.

Let's run the first three and see what they return on this runtime.

In [ ]:
print(f"Running in Colab: {iri.is_colab()}")
print(f"GPU attached:     {iri.has_gpu()}")
print(f"Cache directory:  {iri.cache_dir()}")

### The `setup()` wrapper

Calling those helpers by hand at the top of every notebook is repetitive. The library wraps them in a single function — `setup()` — that runs all four (including the Drive mount on Colab) and prints a status summary so you can see what it found.

`setup()` takes one parameter worth knowing about: `gpu_required`. When set to `True`, the function raises an error if no GPU is detected, stopping the notebook before you waste time on a session that can't run. When `False`, it prints a note and continues. This orientation notebook does no model training, so we set `gpu_required=False` — later notebooks (R2-Q1 onward) will set it to `True`.

In [ ]:
iri.setup(gpu_required=False)

The environment is ready. The next section fetches the orientation slice of PlantVillage and shows you what's inside.

### Loading the data

The loader function fetches a curated slice of PlantVillage from a versioned release on GitHub, verifies the file's checksum against a known hash, extracts it into the cache directory we set up in the previous section, and returns a dictionary with three things to work with.

The download happens once. On first run, you'll see progress messages — fetching, verifying, extracting. On subsequent runs in the same session (or in any later session, as long as the cache survives), the function notices the data is already there and returns immediately.

In [ ]:
data = iri.load_plantvillage_orientation()

What did we get back? A dictionary. Let's see its keys.

In [ ]:
data.keys()

### What's in the dictionary

`data['manifest']` is a 38-row pandas DataFrame describing the **full** PlantVillage dataset. One row per class. The full dataset has roughly 54,000 images across those 38 classes; this DataFrame tells us about every one of them, even though our download is a small sample. Section 6 uses the manifest to map the dataset's structure.

`data['sample_paths']` is a 190-row pandas DataFrame, where each row points to one image file actually on disk. Five images per class, chosen deterministically — the same images every time the tarball is built. Section 5 uses `sample_paths` to open and display actual leaves.

`data['sample_dir']` is the directory those 190 image files live in. Most of the time you'll work through `sample_paths` rather than the directory directly, but it's there if you need it.

Let's look at the manifest's structure.

In [ ]:
data['manifest'].head()

Five columns: `class` (the class label), `host` (the plant species), `disease` (the disease name, or `healthy` for the no-disease classes), `is_healthy` (a boolean flag for convenience), and `n_total` (the count of images in that class in the full dataset).

The asymmetry between the manifest and `sample_paths` is worth flagging before we move on. The manifest has 38 rows describing roughly 54,000 images; `sample_paths` has 190 rows describing 190 image files actually on your disk. Both are right — they describe different things. Total class counts and per-class image counts come from the manifest, because they're facts about the full dataset. Visual inspection comes from `sample_paths`, because it points to bytes you can actually open.

Section 5 uses `sample_paths`. Section 6 uses the manifest. One ~3 MB tarball, two complementary views.

## Looking at the leaves

We have 190 image files on disk — five from each of the 38 classes. Let's pick eight of them, one from each of the first eight classes, and put them on screen.

In [ ]:
to_display = data['sample_paths'].groupby('class').head(1).head(8)
to_display

Eight rows, eight images. We'll open each image and lay them out in a 2×4 grid. The class names follow PlantVillage's `Host___Disease` format with triple underscores between host and disease — you'll see those in the titles.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
paths = to_display['path'].tolist()
classes = to_display['class'].tolist()

for ax, path, cls in zip(axes.flat, paths, classes):
    ax.imshow(Image.open(path))
    ax.set_title(cls, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Now look at the backgrounds

You probably looked at the leaves first. That's where the eye goes — they're the foreground, they're what each label names. Look again, but this time at the backgrounds.

The backgrounds are mostly solid — gray, dark gray, near-black. At a glance they look uniform. But class to class, they're not quite the same. Different shades. Different lighting. Some have a faint tabletop edge. Some are slightly warmer or cooler. The variation is subtle, but it's there — and it has nothing to do with the disease the label names.

This is what Section 2 was pointing at. Noyan (2022) showed that a classifier could predict PlantVillage class labels from just eight background pixels at about 49% accuracy. You've now seen what that classifier was reading: not the leaves, but the systematic background variation between classes. The signal isn't hidden — it just isn't where you'd look first.

The cleanness of the photography is what makes PlantVillage so trainable. Solid backgrounds, controlled lighting, single-leaf framing strip out every confound a real-world image would carry. That same cleanness is what makes the backgrounds informative about the labels. The dataset is clean because of how it was made, and informative in the wrong places for the same reason. One fact, two faces.